In [ ]:
# import os
# from dotenv import load_dotenv

# load_dotenv()  # tenta carregar .env novamente (caso exista)

# print("OPENAI_API_KEY existe?", "OPENAI_API_KEY" in os.environ)
# print("Valor da chave (primeiros 7 + últimos 4 chars):")
# key = os.getenv("OPENAI_API_KEY")
# if key:
#     print(f"{key[:7]}...{key[-4:]}")
# else:
#     print("Nenhuma chave encontrada!")

In [ ]:
from app.retrieval.retriever import retrieve, get_relevant_documents

In [ ]:
query_example = "What is the input parameters for createMOI"

In [ ]:
# Test SelfQuery mode
result = retrieve(query=query_example, k=5)

# Show the returned result
print("Generated semantic query:", result["generated_query"])
print("Generated filter:", result["generated_filter"])
print(f"\nFound {len(result['docs'])} documents:\n")

for i, doc in enumerate(result["docs"], 1):
    print(f"[{i}]")
    print(f"release: {doc.metadata.get('release')}")
    print(f"series: {doc.metadata.get('series')}")
    print(f"spec: {doc.metadata.get('spec')}")
    print(f"chunk_index: {doc.metadata.get('chunk_index')}")
    print(f"text preview:\n{doc.page_content}\n")
    print("-" * 80)

In [ ]:
# Test LangGraph compatibility (only docs)
docs = get_relevant_documents(query_example, k=5)
print(f"\nDocs returned for LangGraph: {len(docs)}")
print("First doc preview:", docs[0] if docs else "No docs")

In [ ]:
docs[0]

In [ ]:
result_manual = retrieve(
    query=query_example,
    k=5,
    filters={"release": "Rel-18"}
)

print("Manual filter mode:")
print("Generated query:", result_manual["generated_query"])
print("Filter applied:", result_manual["generated_filter"])
print(f"Found {len(result_manual['docs'])} docs")

In [ ]:
from qdrant_client.http.models import Filter, FieldCondition, MatchValue
from app.ingest.qdrant_factory import QdrantFactory
from app.utils.settings import COLLECTION_NAME

factory = QdrantFactory(device="cpu")
client = factory.client

# Conta quantos pontos têm release="Rel-18" (sem prefixo payload.)
count = client.count(
    collection_name=COLLECTION_NAME,
    count_filter=Filter(
        must=[
            FieldCondition(key="release", match=MatchValue(value="Rel-18"))
        ]
    )
)

print(f"Documents with release='Rel-18': {count.count}")

In [ ]:
from app.ingest.qdrant_factory import QdrantFactory
from app.utils.settings import COLLECTION_NAME

factory = QdrantFactory(device="cpu")
client = factory.client

# Total de pontos na coleção
total_count = client.count(collection_name=COLLECTION_NAME)
print(f"Total points in collection '{COLLECTION_NAME}': {total_count.count}")

# Pega 3 pontos reais (com payload completo)
points, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=3,
    with_payload=True,
    with_vectors=False
)

if points:
    print("\nExemplo de payloads reais (metadados salvos):")
    for point in points:
        print(point.payload)
        print("-" * 60)
else:
    print("Nenhum ponto encontrado na coleção (vazia ou erro ao scroll)")

In [ ]:
from collections import Counter
from app.ingest.qdrant_factory import QdrantFactory

factory = QdrantFactory(device="cpu")
client = factory.client

# Pega até 200 pontos (para não travar se forem muitos)
points, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=200,
    with_payload=True
)

releases = [point.payload.get("release", "MISSING") for point in points]

print("Contagem de valores de 'release' (primeiros 200 pontos):")
print(Counter(releases))

print(f"\nTotal pontos analisados: {len(points)}")
print(f"Total pontos na coleção: {client.count(collection_name=COLLECTION_NAME).count}")


In [ ]:
from app.utils.chunking import load_chunks
from app.utils.settings import CHUNKS_FILE

chunks = load_chunks(CHUNKS_FILE)
print(f"Total chunks carregados do pickle: {len(chunks)}")

if chunks:
    print("\nChaves do primeiro chunk:")
    print(list(chunks[0].keys()))
    
    print("\nTem 'release'? ", "release" in chunks[0])
    print("Valor de release no primeiro chunk:", chunks[0].get("release", "NÃO EXISTE"))
    
    # Verifica nos primeiros 10
    has_release = any(chunk.get("release") for chunk in chunks[:10])
    print("\nAlgum dos primeiros 10 chunks tem 'release' não vazio? ", has_release)